<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/SPARK_SQL_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set up

In [ ]:
from pyspark.sql import SparkSession
import json

if not 'spark' in globals():
  spark = SparkSession.builder.getOrCreate()

We will import the files needed for the exercises as follows:

In [ ]:
%%capture
!wget -O "country.json" "https://mydisk.cs.upc.edu/s/sqLb8CYFzJXg4S4/download/country.json"
!wget -O "laureate.json" "https://mydisk.cs.upc.edu/s/n9jE7AkCyjz4HpG/download/laureate.json"
!wget -O "prize.json" "https://mydisk.cs.upc.edu/s/2DNiLz5ePc934PJ/download/prize.json"

# Warm up


Before doing the exercises, we present a couple of examples that should serve you as reference on how to proceed with the exercises. Precisely, we will process data related
to the Nobel prizes as obtained from their [API](https://nobelprize.readme.io/docs/getting-started). Data provided by such API has JSON format, please make sure to get familiar with the format if you are not familiar with it.


## Example 1

The goal of this example is to show you how to, using PySpark, execute SQL queries over unstructured data.

We will proceed to read the file and print the inferred schema of `country.json`

In [ ]:
fileDF = spark.read.json("/content/country.json")
fileDF.printSchema()
# We will create a temporal view for the DataFrame `fileDF`
fileDF.createOrReplaceTempView("countriesArray")

root
 |-- countries: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- code: string (nullable = true)
 |    |    |-- name: string (nullable = true)



In [ ]:
fileDF.show()

+--------------------+
|           countries|
+--------------------+
|[{DE, Alsace, the...|
+--------------------+



In [ ]:
#The LATERAL VIEW clause will create a new row for each element in the array "countries".
#Precisely, it will generate a struct called countryInfo (see the generated schema printed in line 13)
countriesDF = spark.sql("SELECT countryInfo FROM countriesArray LATERAL VIEW explode(countries) AS countryInfo")
countriesDF.printSchema()

root
 |-- countryInfo: struct (nullable = true)
 |    |-- code: string (nullable = true)
 |    |-- name: string (nullable = true)



In [ ]:
countriesDF.show()

+--------------------+
|         countryInfo|
+--------------------+
|{DE, Alsace, then...|
|        {DE, Alsace}|
|       {DE, Germany}|
|     {AR, Argentina}|
|     {AU, Australia}|
|       {AT, Austria}|
|       {BE, Belgium}|
|         {MM, Burma}|
|       {MM, Myanmar}|
|        {CA, Canada}|
|         {CL, Chile}|
|         {CN, China}|
|      {CO, Colombia}|
|    {CR, Costa Rica}|
|{CZ, Czechoslovakia}|
|{VN, Democratic R...|
|       {DK, Denmark}|
|    {TL, East Timor}|
|         {EG, Egypt}|
|{DE, Federal Repu...|
+--------------------+
only showing top 20 rows



Now, we can query back this dataframe.

In [ ]:
countriesDF.createOrReplaceTempView("countries")
#The function collect_list is an aggregation function that returns a list for an attribute in the group by set
res = spark.sql("SELECT countryInfo.code, collect_list(countryInfo.name) FROM countries GROUP BY countryInfo.code")
res.show()

+----+------------------------------+
|code|collect_list(countryInfo.name)|
+----+------------------------------+
|  MM|          [Burma, Myanmar, ...|
|  LT|          [Lithuania, Russi...|
|  DZ|                     [Algeria]|
|  FI|          [Finland, Russian...|
|  AZ|          [Russian Empire (...|
|  UA|          [Austria-Hungary ...|
|  RO|                     [Romania]|
|  ZM|          [Northern Rhodesi...|
|  NL|             [the Netherlands]|
|  PL|          [Poland, Poland, ...|
|  MK|          [Ottoman Empire (...|
|  MX|                      [Mexico]|
|  CN|                [China, Tibet]|
|  AT|          [Austria, Austria...|
|  RU|          [Russia, USSR, Pr...|
|  HR|          [Austria-Hungary ...|
|  CZ|          [Czechoslovakia, ...|
|  PT|                    [Portugal]|
|  GH|          [Ghana, Gold Coas...|
|  LR|                     [Liberia]|
+----+------------------------------+
only showing top 20 rows



# Exercise 1

Using Spark, return for each year, the total number of laureates that got a Nobel prize. An excerpt of your output should look like the following (not necessarily ordered):
* 2017 -- 12
* 2016 -- 11
* 2015 – 11

In [ ]:
# TODO: Replace <FILL IN> with appropriate code
fileDF = spark.read.json('/content/prize.json')
fileDF.createOrReplaceTempView("prizesArray")
res = spark.sql("<FILL IN>")
res.show()

+----+--------+
|year|count(e)|
+----+--------+
|2017|      12|
|2016|      11|
|2015|      11|
|2014|      13|
|2013|      13|
|2012|      10|
|2011|      13|
|2010|      11|
|2009|      13|
|2008|      12|
|2007|      12|
|2006|       9|
|2005|      13|
|2004|      12|
|2003|      11|
|2002|      13|
|2001|      15|
|2000|      13|
|1999|       7|
|1998|      12|
+----+--------+
only showing top 20 rows



# Exercise 2

Using Spark, return the name and date of birth of those laureates that got a Nobel prize motivated by their work on DNA. An excerpt of your output should look like the following:
* Berg, Paul -- 1926-06-30
* Lindahl, Tomas -- 1938-01-28
* Modrich, Paul -- 1946-06-13
* Sancar, Aziz -- 1946-09-08

In [ ]:
# TODO: Replace <FILL IN> with appropriate code
fileDF = spark.read.json("/content/laureate.json")
fileDF.createOrReplaceTempView("laureatesArray")
res = spark.sql(<FILL IN>)
res.show()

+---------+-------+----------+
|firstname|surname|      born|
+---------+-------+----------+
|     Paul|   Berg|1926-06-30|
|    Tomas|Lindahl|1938-01-28|
|     Paul|Modrich|1946-06-13|
|     Aziz| Sancar|1946-09-08|
+---------+-------+----------+

